# Phase 16b — the fine-tuning run

**This notebook contains no logic.** Every cell installs, invokes, or
downloads. The training code lives in the repo and is pinned to a commit,
so what ran is always recoverable from `git_commit`.

## What makes this different from `smoke_run.ipynb`

14.5 proved the chain works. This one is the first run that can **change a
number**, and it changes exactly one thing against that known-good
configuration: `--augment`. Everything else — `--no-amp`, `--batch-size 2
--accum-steps 4`, `--workers 2` — is held fixed, because five separate
failures in HANDOFF §4 came from moving more than one variable at a time.

It also uses the **full index** (962 train tracks), not the 20-track smoke
subset.

## The target

**Beat 0.787 on MAPS, not 0.969 on MAESTRO.** MAESTRO is ByteDance's own
training distribution; beating it is open research and would not mean what
it appears to mean. Some MAESTRO loss is an acceptable price here — the 20%
clean passthrough is what bounds it.

## Before running
1. **Settings → Accelerator → GPU** (P100 or T4).
2. **Settings → Internet → On** (pip, and the pretrained checkpoint).
3. **Add Data →** search `maestro-v3.0.0` and attach the public dataset.
4. Set `COMMIT` below to the commit you want to run.

In [ ]:
COMMIT = "phase-16b-gpu-run"   # a branch, tag, or full SHA
REPO = "https://github.com/ImSe4n/PTify.git"

# --no-deps is load-bearing: Kaggle's preinstalled torch is much newer than
# this project's local pin (measured: torch 2.10 / numpy 2.0 there vs
# 2.2/1.26 here). That is FINE — a checkpoint is a plain state_dict and
# crosses versions — but letting a resolver loose would reinstall torch and
# waste most of the session.
!pip install -q --no-deps git+{REPO}@{COMMIT}
!pip install -q --no-deps piano_transcription_inference torchlibrosa \
    mido pretty_midi librosa soundfile resampy audioread soxr lazy_loader msgpack

import importlib
missing = []
for m in ["mido", "pretty_midi", "librosa", "soundfile", "resampy", "soxr",
          "torchlibrosa", "piano_transcription_inference"]:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append(f"{m}: {type(e).__name__} {e}")
print("MISSING:", missing or "none — all imports OK")

import torch, numpy
print("torch", torch.__version__, "| numpy", numpy.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Clone the repo and find the MAESTRO mount

The segment index stores **relative** paths, so it only needs the directory
those resolve against.

In [ ]:
# Clone as well as pip-install: pip installs the PACKAGES but not the repo's
# data files, and the segment index in benchmarks/ is a data file.
!git clone -q --depth 1 --branch {COMMIT} {REPO} /kaggle/working/PTify || true
!ls -la /kaggle/working/PTify/benchmarks/*.json

import glob, os
print("\n--- attached datasets ---")
for path in sorted(glob.glob("/kaggle/input/*/")):
    print(path)
    for sub in sorted(glob.glob(path + "*"))[:6]:
        print("   ", os.path.basename(sub))

In [ ]:
# Verified working with the `alonhaviv/the-maestro-dataset-v3-0-0` public
# dataset. Kaggle nests an attached dataset under the uploader's name, so this
# path is specific to that upload. What matters is that the directory CONTAINS
# the year folders (2004/ ... 2018/).
AUDIO_ROOT = "/kaggle/input/datasets/alonhaviv/the-maestro-dataset-v3-0-0/maestro-v3.0.0"

# The FULL index this time, not the smoke subset: 962 train / 137 validation
# tracks, 632,783 segments.
INDEX = "benchmarks/maestro_segments.json"

import json, pathlib
index = json.load(open(f"/kaggle/working/PTify/{INDEX}"))
print("summary:", json.dumps(index["summary"], indent=2))

sample = index["tracks"][0]["audio_filename"]
probe = pathlib.Path(AUDIO_ROOT) / sample
print("\nprobe:", probe)
print("EXISTS:", probe.exists(), "<- must be True before going further")
print("MIDI :", (pathlib.Path(AUDIO_ROOT) / index["tracks"][0]["midi_filename"]).exists())

## 2. Train

`--resume auto` starts fresh when there is no checkpoint and continues when
there is, so **re-running this cell after a session dies is the recovery
procedure** — there is no separate resume path to get wrong.

### Why each flag is what it is

- **`--augment`** — the point of the run. Continuous room/detune drawn per
  segment, hash-seeded so a resume reproduces it exactly. Targets the measured
  18.3-point MAESTRO→MAPS gap, of which 12.9 is room acoustics alone.
- **`--no-amp`** — load-bearing. With AMP on, the near-OOM state produced NaN
  in all four heads at step 0 rather than an honest allocation failure. fp32
  fails loudly instead. AMP is a later optimisation, attempted only once an
  augmented run is known good.
- **`--batch-size 2 --accum-steps 4`** — effective batch 8 at a quarter of the
  memory. Batch 8 OOMs on a T4: the model runs **four parallel CRNN branches**
  over 1001×229 features. The accumulated gradient is provably identical to
  the full-batch one.
- **`--steps 10000`** — ~10.3h at the measured 0.27 steps/s, sized to one
  session. Note this is only **15% of one epoch** (a full epoch is 70,517
  steps ≈ 72h), which is fine: segments overlap by 90%, so one partial pass
  already covers the unique audio well.
- **`--save-every-seconds 1800`** — wall clock, because Kaggle kills at a fixed
  hour regardless of step count.
- **`--validate-every 500`** — logs BOTH `val_*` (clean, the regression guard)
  and `val_aug_*` (what this run optimises). Watch for clean val degrading
  while augmented val improves; that is trading away what already works.

If it OOMs anyway, drop to `--batch-size 1 --accum-steps 8`. The error message
tells you the exact flags.

In [ ]:
!cd /kaggle/working/PTify && python -m training.train \
    --index benchmarks/maestro_segments.json \
    --audio-root {AUDIO_ROOT} \
    --out /kaggle/working/checkpoints \
    --augment \
    --augment-seed 0 \
    --device cuda \
    --no-amp \
    --steps 10000 \
    --batch-size 2 \
    --accum-steps 4 \
    --workers 2 \
    --log-every 50 \
    --validate-every 500 \
    --val-batches 20 \
    --save-every-seconds 1800 \
    --keep-checkpoints 2 \
    --resume auto

## 3. Read the curves

Two things to check, and the second is the one that matters:

1. **`total` falls.** If it does not, something is wrong with the run itself.
2. **`val_aug_total` falls faster than `val_total` rises.** `val_total` is the
   clean regression guard — it may drift up a little, and that is the price
   being paid. `val_aug_total` is the metric this whole track exists to move.

A clean val curve improving while the augmented one goes nowhere would mean
the augmentation is not teaching anything, and would only surface at MAPS
scoring otherwise.

In [ ]:
import json
log = [json.loads(l) for l in open("/kaggle/working/checkpoints/train_log.jsonl")]
train_rows = [r for r in log if "total" in r]
val_rows = [r for r in log if "val_total" in r]

print("steps logged :", train_rows[0]["step"], "->", train_rows[-1]["step"])
print("loss         :", round(train_rows[0]["total"], 4), "->",
      round(train_rows[-1]["total"], 4))
print("steps/s      :", [r["steps_per_s"] for r in train_rows][-3:])
print("peak GPU GB  :", max(r.get("gpu_mem_gb", 0) for r in train_rows))

print("\nstep     VAL(clean)  AUG")
for r in val_rows:
    print(f"{r['step']:>6}   {r['val_total']:.4f}      "
          f"{r.get('val_aug_total', float('nan')):.4f}")

# After a resume the step counter CONTINUES rather than restarting.
print("\ncontinued past a restart:",
      train_rows[-1]["step"] > len(train_rows))
print("epoch at the end:", train_rows[-1]["epoch"], "(1 is expected — one",
      "epoch is 70,517 steps)")

## 4. The kill/resume drill

**Interrupt the training cell manually partway through**, then run it again.
The log must show `Resumed from step_N.pt (epoch E)` and the step counter must
continue rather than restart. A 12-hour cap makes resume the only way a
10-hour run ever finishes.

The epoch in that message matters: a resume must continue **in the epoch it
was interrupted in**. Resuming at epoch 1 when the run was in epoch 5 would
silently re-draw epoch 1's augmentation for the rest of the run.

## 5. Verify the artifact BEFORE trusting any score

`PianoTranscription` re-downloads any checkpoint under 160MB and loads with
`strict=False`. Either failure produces a plausible number from **ByteDance's**
weights under your filename. `assert_deployable` is what stands between this
run and a fabricated result.

In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working/PTify")
from training.model import assert_deployable

ckpt = "/kaggle/working/checkpoints/ptify-note-pedal.pth"
assert_deployable(ckpt)
print("OK: %.1f MB" % (os.path.getsize(ckpt) / 1e6))

## 6. Download it

Kaggle output persistence has bitten people; get the artifact off-box before
the session ends. Losing 10 GPU-hours to a lost output is the failure this
guards against.

Then, locally — this is the actual Phase 16b gate:

```bash
python -c "from training.model import assert_deployable; assert_deployable('ptify-note-pedal.pth')"

# MAPS — the real scoreboard. ~1.8h at ~1.87x real time.
set PYTHONUNBUFFERED=1
python -m evaluation --audio-dir recordings/maps_paired \
    --engine bytedance --preset clean \
    --checkpoint ptify-note-pedal.pth \
    --json benchmarks/real/maps-paired-ptify-clean.json

# MAESTRO — the regression check. Some loss here is an acceptable price.
python -m evaluation --audio-dir recordings/maestro_test12 \
    --engine bytedance --preset clean \
    --checkpoint ptify-note-pedal.pth \
    --json benchmarks/real/maestro-ptify-clean.json
```

then diff each against its baseline with `report.compare_reports()`, which
joins on `(engine, case, preset)` — never by position.

In [ ]:
from IPython.display import FileLink
FileLink("/kaggle/working/checkpoints/ptify-note-pedal.pth")